# 07 · Sensor streams and uncertainty

**Question:** Can an uncertain model track a changing urban heat signal from noisy observations?

**Runtime:** Python / Pyodide in JupyterLite. Choose **Run → Run All Cells**.
Each notebook is independent; start with a fresh kernel. Initial runtime/package downloads need internet.
The initial baseline uses **synthetic educational fixtures** near a Gaborone-area anchor.
Extended investigations explicitly identify bundled real OSM, Sentinel or terrain extracts where used;
these do not validate the synthetic population, mobility, exposure or scenario assumptions.

**Astra experiment:** Ask Astra to diagnose an outlier and missing readings and justify whether uncertainty should increase.
Run the numerical baseline first, then give Astra the exported evidence and the prompt below.
The baseline does not call or run GPT-6 Astra. No API key is needed.


In [ ]:
import sys, json, math, time
from pathlib import Path
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "matplotlib"])
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from twin import *
style()
print("Runtime:", sys.platform, "| data: synthetic | seed: 42")


## A one-dimensional Kalman filter
One representative location has a random-walk latent signal. A sensor occasionally drops readings and has one large outlier.
The filter rejects readings beyond four predicted standard deviations. This is a teaching model, not a citywide spatial filter.


In [ ]:
rng=np.random.default_rng(42); steps=80; process_var=.12; sensor_var=1.5
truth=30+np.cumsum(rng.normal(0,np.sqrt(process_var),steps))
observed=truth+rng.normal(0,np.sqrt(sensor_var),steps)
observed[20:28]=np.nan; observed[45]+=15
estimate=[]; variance=[]; rejected=[]; mean=30.; var=4.
for t,obs in enumerate(observed):
    var+=process_var
    if np.isfinite(obs):
        innovation=obs-mean
        if abs(innovation)<=4*np.sqrt(var+sensor_var):
            gain=var/(var+sensor_var); mean+=gain*innovation; var*=(1-gain)
        else: rejected.append(t)
    estimate.append(mean); variance.append(var)
estimate=np.array(estimate); variance=np.array(variance)
assert (variance>0).all() and 45 in rejected
assert variance[27]>variance[19]
rmse=float(np.sqrt(np.mean((estimate-truth)**2)))
print('Filter RMSE:',rmse,'| rejected indices:',rejected)


In [ ]:
t=np.arange(steps); fig,ax=plt.subplots()
ax.fill_between(t,estimate-1.96*np.sqrt(variance),estimate+1.96*np.sqrt(variance),alpha=.2,label='Approximate 95% model interval')
ax.plot(t,truth,label='Synthetic truth'); ax.plot(t,estimate,label='Estimate')
ax.scatter(t,observed,s=12,c='#e45756',label='Sensor')
ax.set(xlabel='Time step',ylabel='Synthetic degrees C',title='Assimilation with missing and rejected readings'); ax.legend(); plt.show()
export_json('07_assimilation.json',dict(synthetic=True,rmse=rmse,rejected=rejected,process_variance=process_var,sensor_variance=sensor_var))


## Teaching lab: Diagnose uncertainty
**Before using AI:** write a prediction, run the numerical baseline, and explain one surprising result.

**Astra evidence:** send only 07_assimilation.json and the sensor plot. Use a fresh conversation and record the exact model,
date, interface, tools and prompt. A task with a supplied result tests interpretation, not independent calculation.

**Copyable Astra prompt:**
> Explain the missing interval and rejected outlier. Distinguish a model uncertainty band from empirically verified interval coverage.

**Implementation brief — review the runnable extension below:**
> Run the filter on multiple seeded streams and measure empirical interval coverage against synthetic truth. Edit the authoring source in `scripts/make_notebooks.py` (or
> `scripts/teaching_labs.py` for labs 11–13), regenerate the notebooks and preserve browser compatibility.
> Use NumPy, Matplotlib and the standard library. Include a small reference case that can be checked by hand.

**Acceptance evidence:** Missing readings increase prediction variance; changing sensor noise changes gain; report coverage without forcing it to 95%.

The extended investigation later in this notebook implements this line of work with maps and checks.
Use the brief to review the implementation or ask Codex for a further controlled variation.

**Share back:** save your original prediction, changed parameter, result, model response and one limitation.
Download the edited notebook and exports. If you have no model access, exchange explanations with a partner
and evaluate them with the same criteria; do not label that activity as an Astra trial.


## Extended investigation: Map stale sensors and test uncertainty on held-out streams

Six mapped sensors have noisy signals, inserted gaps and outliers. Validate timestamp order, track time since the last accepted observation, and compare interval coverage on 30 independently seeded streams under correctly specified and underestimated observation noise. Coverage is measured rather than forced to 95%; rejection and finite samples can change it.

**Read the concept map first:** identify the input, method, evidence and limit. Predict the spatial pattern before running the comparison.


In [ ]:
from extension_tools import concept_map, export_investigation
concept_map('Map stale sensors and test uncertainty on held-out streams', ['Timestamped sensor sites', 'Reject / predict / update', 'Coverage + staleness maps', 'Model intervals need calibration'])
plt.show()


In [ ]:
from extension_tools import kalman_stream, timestamp_check
ex_times=np.datetime64('2026-01-01T00:00')+np.arange(80)*np.timedelta64(15,'m')
assert timestamp_check(ex_times) and not timestamp_check(ex_times[::-1])
ex_sites=city(seed=91,n=6)['xy']; ex_truths=[]; ex_observed=[]; ex_estimates=[]; ex_variances=[]; ex_ages=[]
for site in range(6):
    rng=np.random.default_rng(100+site); truth=30+np.cumsum(rng.normal(0,np.sqrt(.12),80)); observed=truth+rng.normal(0,np.sqrt(1.5),80)
    observed[20:28+site]=np.nan; observed[45]+=15
    mean,var,age=kalman_stream(observed); ex_truths.append(truth); ex_observed.append(observed); ex_estimates.append(mean); ex_variances.append(var); ex_ages.append(age)
ex_coverage={}
for assumed in [.15,1.5,6.]:
    cover=[]
    for seed in range(1000,1030):
        rng=np.random.default_rng(seed); truth=30+np.cumsum(rng.normal(0,np.sqrt(.12),80)); obs=truth+rng.normal(0,np.sqrt(1.5),80); obs[20:28]=np.nan
        mean,var,_=kalman_stream(obs,assumed_sensor_var=assumed); cover.append(float(np.mean(abs(mean-truth)<=1.96*np.sqrt(var))))
    ex_coverage[str(assumed)]=cover
fig,axes=plt.subplots(1,3,figsize=(14,4)); ex_snapshot=30
dots=axes[0].scatter(*ex_sites.T,c=np.array(ex_ages)[:,ex_snapshot],s=150,cmap='magma')
for i,xy in enumerate(ex_sites): axes[0].annotate(f'S{i}',xy,xytext=(5,5),textcoords='offset points')
map_axes(axes[0],'Time since last accepted reading'); fig.colorbar(dots,ax=axes[0],label='15-minute steps')
ex_errors=np.array(ex_estimates)-np.array(ex_truths)
image=axes[1].imshow(ex_errors,aspect='auto',cmap='coolwarm',vmin=-3,vmax=3); axes[1].set(title='Spatial sensor residuals',xlabel='Time step',ylabel='Sensor ID'); fig.colorbar(image,ax=axes[1],label='Synthetic °C error')
axes[2].boxplot(list(ex_coverage.values()),tick_labels=list(ex_coverage)); axes[2].axhline(.95,ls='--',c='gray'); axes[2].set(xlabel='Assumed sensor variance',ylabel='Empirical interval coverage',title='30 held-out streams, true variance 1.5',ylim=(0,1.05))
fig.tight_layout(); extension_figure=fig
extension_evidence=dict(coverage={k:float(np.mean(v)) for k,v in ex_coverage.items()},ages_at_step_30=np.array(ex_ages)[:,30].tolist(),interval='mean ± 1.96 standard deviations',step_minutes=15)
extension_expected=dict(stalest_sensor=int(np.argmax(np.array(ex_ages)[:,30])),max_staleness_minutes=int(np.max(np.array(ex_ages)[:,30])*15))
extension_task='Return stalest_sensor (zero-based ID) and max_staleness_minutes from the provided age vector. Age measures accepted observations, not merely a sensor transmission.'


In [ ]:
extension_prompt,extension_reference=export_investigation('07',extension_evidence,extension_expected,extension_task,extension_figure)
plt.show()
print('Send only 07_extension_prompt.json and optionally 07_extension_map.png to Astra. Keep the reference in this kernel.')


### Ask, check and explain the spatial result
Download `07_extension_prompt.json` and `07_extension_map.png`. Send the evidence in a fresh Astra
session with the exact task and output fields. For a separate image-only experiment, supply only the image
and task wording, record that condition, and allow clarification if the image does not contain enough information.
Never send the reference or this complete notebook for a blind trial.

Paste the unedited response below or upload a UTF-8 response file. The default is deliberately empty:
**no model trial has run**. After recording a real answer, set `extension_source='MODEL'` and complete the metadata.
The numeric tolerance is 0.01 in each requested field's declared units; labels and booleans require exact agreement.
Keep that rule fixed before examining a response. Inspect the explanation separately for unsupported claims.

**Visual reflection:** point to one map pattern, cite the numerical check that supports it, and name one thing
the visual cannot establish. Re-run one parameter change and preserve both maps. The implementation above
is available for inspection and modification; it is not a measured claim about model accuracy.


In [ ]:
from experiments import grade_answer
extension_source='NOT RUN'
extension_model=''; extension_date=''; extension_interface=''; extension_condition='text-and-map'
extension_response_file=''; extension_response_text=''
if extension_response_file:
    extension_response_text=Path(extension_response_file).read_text(encoding='utf-8')
if extension_source=='MODEL':
    assert extension_model and extension_date and extension_interface and extension_response_text, 'Record the actual model, date, interface and response'
extension_tolerances={key:.01 for key,value in extension_reference.items() if type(value) in (int,float)}
extension_grade=grade_answer(extension_response_text,extension_reference,extension_tolerances) if extension_response_text else dict(status='NOT RUN')
display(extension_grade)
export_json('07_extension_trial.json',dict(source=extension_source,model=extension_model,date=extension_date,
    interface=extension_interface,condition=extension_condition,prompt=extension_prompt,response_text=extension_response_text,
    tolerances=extension_tolerances,reference=extension_reference,grade=extension_grade))


## Save the investigation
Download the edited notebook, evidence, map and trial record from `exports/`. Browser storage does not
commit changes to GitHub. Keep data attribution and the distinction between observed data, synthetic
assumptions and actual model responses when sharing. The extended code and its checks above are part of
this notebook; use them as the starting point for your next controlled comparison.
